<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gecko-Academy/dev3pack-cohort-2026-09/blob/main/units/en/unit2/capstone/notebook.ipynb)


# Capstone: the source-grounded research assistant

The project, proved by four gates and one thing you write: a cited answer, a refusal that costs no model call, a readable trace, a green eval, and the ranked list of what it still does badly.

You do not build the capstone in this notebook. You build it between sessions, from week 2 onward; this is where you run it and read the verdict.

In [1]:
# Preflight: environment checks with a fix for anything missing. It never raises.
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "src"))
CORPUS_DIR = REPO_ROOT / "data" / "corpus"

try:
    from bootcamp_agent.preflight import preflight
except ImportError:
    if "google.colab" in sys.modules:
        # Colab starts in /content with no course in it, so fetch one. A shallow
        # clone of the COHORT repository, which is the public one; the source
        # repository is private and would ask this learner for credentials.
        import subprocess

        target = Path("/content/dev3pack")
        if not (target / "pyproject.toml").exists():
            print("Colab detected — fetching the course (about 20 seconds)…")
            subprocess.run(
                ["git", "clone", "-q", "--depth", "1",
                 "https://github.com/Gecko-Academy/dev3pack-cohort-2026-09.git", str(target)],
                check=True,
            )
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "-e", str(target)], check=True
        )
        REPO_ROOT = target
        sys.path.insert(0, str(REPO_ROOT / "src"))
        import os

        os.chdir(REPO_ROOT)
        from bootcamp_agent.preflight import preflight

        print(f"ready — the course is at {REPO_ROOT}")
    else:
        print("❌ bootcamp_agent not importable -> in the repo root run: uv sync --group dev")
        print("   then pick the .venv kernel (or start Jupyter with: uv run jupyter lab)")
else:
    LIVE = preflight(REPO_ROOT)  # the configured lane's client; FakeLLM whenever the lane is down

✅ Python 3.11 (need >= 3.11)
✅ kernel is the repo .venv
✅ corpus loads (6 documents)
✅ lane = fake (deterministic, offline)
ready. LIVE is the fake lane.


In [2]:
from bootcamp_agent.checks import check, review

## The reference architecture

```
question -> retrieve (lexical, top-k) -> [refuse early]
        -> prompt (context + strict JSON instructions)
        -> LLM (provider seam) -> parse (strict) -> verify citations
        -> AgentResult(answer, trace)
```

Integration rule: every component stays independently testable. Integration adds NO logic, only wiring.

The four checkpoints below are **gates, not exercises**. The code is written; you run it and its check confirms the gate. Come back and rerun them after every increment — the capstone is done when all four are green and the fifth item is written.

## Checkpoint 1: answer one supported question, citations verified

In [3]:
import json

from bootcamp_agent.agent import answer_question
from bootcamp_agent.documents import load_corpus
from bootcamp_agent.llm import FakeLLM

documents = load_corpus(CORPUS_DIR)
client = FakeLLM(responses={
    "chunking": json.dumps({
        "answer": "Chunking splits documents into retrievable passages.",
        "citations": ["rag-basics"], "confidence": 0.9, "needs_human_review": False,
    })
})
supported = answer_question("How does chunking work in RAG?", documents, client)
print(supported.answer.answer)
print(f"citations={list(supported.answer.citations)} review={supported.answer.needs_human_review}")

Chunking splits documents into retrievable passages.
citations=['rag-basics'] review=False


In [4]:
check("cap01-e1", supported)

✅ cap01-e1 passed


True

## Checkpoint 2: reject one unsupported question, BEFORE any model call

In [5]:
probe = FakeLLM()
unsupported = answer_question("qual o placar do jogo de ontem?", documents, probe)
print(unsupported.answer.answer)
print(f"citations={list(unsupported.answer.citations)} model calls={len(probe.calls)}")

I don't know based on the provided corpus.
citations=[] model calls=0


In [6]:
check("cap01-e2", {"result": unsupported, "probe_calls": len(probe.calls)})

✅ cap01-e2 passed


True

## Checkpoint 3: produce a trace a reviewer can follow

In [7]:
for event in supported.trace:
    print(f"[{event.kind:9}] {event.detail}")
trace_kinds = [event.kind for event in supported.trace]

[retrieve ] top_k=3 -> [('rag-basics', 1), ('rag-basics', 2), ('evaluation-basics', 0)]
[llm_call ] attempt 1: 143 chars
[decision ] answered with citations ['rag-basics']


In [8]:
check("cap01-e3", trace_kinds)

✅ cap01-e3 passed


True

## Checkpoint 4: the eval gate. v1 is done when this is green

In [9]:
from bootcamp_agent.evals import format_report, load_cases, run_evals


def grounded_fake():
    def r(doc_id, text):
        return json.dumps({"answer": text, "citations": [doc_id],
                           "confidence": 0.9, "needs_human_review": False})
    return FakeLLM(responses={
        "chunking": r("rag-basics", "Chunking splits documents into passages."),
        "structured outputs": r("structured-outputs", "Validate at the boundary."),
        "stopping conditions": r("agent-loops", "Budgets and defined exits."),
        "mcp server": r("mcp-overview", "Tools execute; skills instruct; MCP serves."),
        "prompt injection": r("prompt-injection", "Layered defenses."),
    })


cases = load_cases(REPO_ROOT / "data" / "evals" / "golden.jsonl")
gate = run_evals(cases, documents, grounded_fake())
print(format_report(gate))

| # | Question | Result | Detail |
|---|---|---|---|
| 1 | How does chunking work in retrieval-augmented generation? | PASS | cited ['rag-basics'] |
| 2 | Why should structured outputs be validated by the applicatio | PASS | cited ['structured-outputs'] |
| 3 | What stopping conditions should an agent loop have? | PASS | cited ['agent-loops'] |
| 4 | What is the difference between a tool, a skill, and an MCP s | PASS | cited ['mcp-overview'] |
| 5 | What defenses help against prompt injection in retrieved con | PASS | cited ['prompt-injection'] |
| 6 | What is the capital of Mars? | PASS | refused as expected |
| 7 | zxqv wubble frobnicate | PASS | refused as expected |
| 8 | Qual foi o placar do jogo de ontem? | PASS | refused as expected |

**8/8 passed** (pass rate 100%)


In [10]:
check("cap01-e4", gate)

✅ cap01-e4 passed


True

## 5. Exercise: the ranked issue list

**Context.** List what the capstone does NOT do well, ranked by impact. This is session 14's hardening backlog and demo day's honest limitation, and writing it now is cheaper than discovering it on stage.

**Instructions.**

1. Rank 1 is filled as an example. Replace it with your own if you disagree.
2. Write ranks 2 and 3: the issue, and its impact. The impact is what ranks it.
3. Run the check. It refuses blanks and refuses ranks with gaps or ties.

In [11]:
issues = [
    {"rank": 1, "issue": "Lexical retrieval misses paraphrases with no word overlap",
     "impact": "a user who asks in their own words gets a refusal for a supported question"},
    {"rank": 2, "issue": "The evaluator checks which doc was cited, not whether the answer is faithful",
     "impact": "a model that cites correctly but fabricates content passes the eval undetected"},
    {"rank": 3, "issue": "Nothing bounds a provider that hangs rather than failing",
     "impact": "a stalled HTTP call blocks the run forever with no timeout refusal"},
]
for row in issues:
    print(f"{row['rank']}. {row['issue'] or '(empty)'}")


1. Lexical retrieval misses paraphrases with no word overlap
2. The evaluator checks which doc was cited, not whether the answer is faithful
3. Nothing bounds a provider that hangs rather than failing


**Expected output** (yours may differ in wording, not in shape):

```
1. Lexical retrieval misses paraphrases with no word overlap
2. The evaluator checks which doc was cited, not whether the answer is faithful
3. Nothing bounds a provider that hangs rather than failing
✅ cap01-e5 passed
```

In [12]:
check("cap01-e5", issues)

✅ cap01-e5 passed


True

## The next increment

Pick rank 1 from your list and write the failing test for it. Session 14's clinic starts from a red test, not from a blank page — and rank 1 is what you would name as your own limitation in minute 6 of the demo.

## Review

The scorecard for this notebook. Every ❌ line names the exercise and the hint.

In [13]:
review("cap01")

cap01: 5/5 passed  ·  500/500 marks


True